# tests to mix the openEO processed eunis probabilities via UDP
This scripts shows how to paramaterize the UDP to retrieve the habitat map from openEO for given year, area. <br>
<br>

In [49]:
import openeo
import os
from time import sleep

In [50]:
# establish connection to OpenEO and authenticate
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


In [60]:
#namespace="/home/smetsb/PycharmProjects/OpenEO-UDP-UDF-catalogue/UDP/json/udp_eunis_mixer_alpha3.json"
namespace="https://raw.githubusercontent.com/ESA-WEED-project/OpenEO-UDP-UDF-catalogue/refs/heads/alpha3_udp/UDP/json/udp_eunis_mixer_alpha3.json"
namespace="https://raw.githubusercontent.com/ESA-WEED-project/OpenEO-UDP-UDF-catalogue/refs/heads/UDP_Obsgession/UDP/json/udp_obsgession_w23_lai.json"

In [86]:
#define parameters
# temporal
start_date = '2025-06-01'
end_date = '2025-07-01'
# temporal binning and aggregator
binning_period = 'month'
temp_aggregator = 'median'
# spatial (has to be a openEO BBOX dict)
AOI = {'east': 4880000, 'south': 2896000, 'west': 4876000, 'north': 2900000, 'crs': 3035}
# resolution and EPSG of output
out_res = 20.0
out_epsg = 3035



## get the habitat map

In [87]:
#get cube from udp
cube = connection.datacube_from_process(
    process_id="udp_obsgession_w23_lai",
    namespace=namespace,
    start_date = start_date,
    end_date = end_date,
    binning_period = binning_period,
    temp_aggregator = temp_aggregator,
    aoi = AOI,
    resolution = out_res,
    epsg = out_epsg
    )

In [88]:
#create and start job
job = cube.create_job(title=f'LAI test', auto_add_save_result=False)
#job.start_job()

In [89]:
job.start_job()

<BatchJob job_id='j-2603131606484c41807c0101d766b0d7'>

In [90]:
#follow-up on job
print(job.job_id)
while job.status() not in ['finished','error','canceled']:
    print(f"going to sleep job not yet done: status : {job.status()}")
    sleep(60)

print(f"Job done: status : {job.status()}")

j-2603131606484c41807c0101d766b0d7
going to sleep job not yet done: status : created
going to sleep job not yet done: status : queued
going to sleep job not yet done: status : queued
going to sleep job not yet done: status : queued
going to sleep job not yet done: status : queued
going to sleep job not yet done: status : running
going to sleep job not yet done: status : running
Job done: status : finished


In [85]:
#get results (metadata)

job.describe()

{'costs': 4,
 'created': '2026-03-13T14:50:26Z',
 'id': 'j-26031314502646e9956a7d023e2c4dd7',
 'process': {'process_graph': {'udpobsgessionw23lai1': {'arguments': {'aoi': {'crs': 3035,
      'east': 4880000,
      'north': 2900000,
      'south': 2896000,
      'west': 4876000},
     'binning_period': 'month',
     'end_date': '2025-07-01',
     'epsg': 3035,
     'resolution': 20.0,
     'start_date': '2025-06-01',
     'temp_aggregator': 'mean'},
    'namespace': 'https://raw.githubusercontent.com/ESA-WEED-project/OpenEO-UDP-UDF-catalogue/refs/heads/UDP_Obsgession/UDP/json/udp_obsgession_w23_lai.json',
    'process_id': 'udp_obsgession_w23_lai',
    'result': True}}},
 'progress': 100,
 'status': 'finished',
 'title': 'LAI test',
 'updated': '2026-03-13T14:53:12Z',
 'usage': {'duration': {'unit': 'seconds', 'value': 98},
  'input_pixel': {'unit': 'mega-pixel', 'value': 5.976029396057129}}}

In [17]:
#get errors
job.logs(level='error')

[{'id': '[1773238465403, 494375]',
  'time': '2026-03-11T14:14:25.403Z',
  'level': 'error',
  'message': 'OpenEO batch job failed: OpenEOApiException(status_code=500, code=\'Internal\', message="Unexpected error during \'aggregate_temporal_period\': Py4JError(\'An error occurred while calling o6323.expressionStart. Trace:\\\\npy4j.Py4JException: Method expressionStart([class java.util.HashMap, class java.util.HashMap]) does not exist\\\\n\\\\tat py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:321)\\\\n\\\\tat py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:329)\\\\n\\\\tat py4j.Gateway.invoke(Gateway.java:274)\\\\n\\\\tat py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)\\\\n\\\\tat py4j.commands.CallCommand.execute(CallCommand.java:79)\\\\n\\\\tat py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184)\\\\n\\\\tat py4j.ClientServerConnection.run(ClientServerConnection.java:108)\\\\n\\\\tat java.base/java.lang.Thread.run(Thread.java:1583)\\\\n\\\\n\'). The process had these arguments: {\'data\': GeopysparkDataCube(metadata=GeopysparkCubeMetadata(dimension_names=[\'x\', \'y\', \'t\', \'bands\'], band_names=[\'LAI\'])), \'period\': \'month\', \'reducer\': {\'process_graph\': {\'median1\': {\'process_id\': {\'from_parameter\': \'temp_aggregator\'}, \'arguments\': {\'data\': {\'from_parameter\': \'data\'}}, \'result\': True}}}, \'temp_aggregator\': \'mean\'} ", id=\'no-request\')'}]